# V5 여러 그림 — 나란히 놓으면 각자 자를 든다 · 실습 (T5)

> **6단계 프레임: ⑤확인** — 그림 둘을 견주기 전에 **두 축이 같은 자인지** 봅니다.

**이 노트북의 목표**

1. `plt.subplots(행, 열)`로 Figure 하나에 **Axes 여럿**을 만들고, 자리 번호로 꺼낸다
2. 막대 그래프의 **자동 축 천장**이 어디서 오는지 손으로 계산한다
3. **세트피스** ⭐ — 나란히 놓은 두 그림에서 무엇이 어긋나는지 표로 따라간다
4. 두 축을 **같은 자**로 만드는 법과, 그렇게 보이지만 아닌 길을 본다
5. 격자와 **위아래 배치**를 쓰고, 여러 그림에서 나는 **오류 다섯 가지**를 읽는다

**빈칸은 `___` 입니다.** 총 **6개**.

## Part A. 준비 — 서른 날을 앞·뒤 보름으로

V4에서 만든 그 표입니다. 이 셀은 **그대로 실행만 하세요.**

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

rentals = [14230, 15010, 13120, 13540, 6880, 12760, 13980, 16420, 15870, 13310,
           13650, 9240, 14020, 14890, 16060, 5120, 7340, 10210, 4050, 8760,
           12480, 5430, 5970, 11390, 9880, 11390, 4610, 7890, 13240, 13880]
temp = [24.1, 23.5, 22.8, 24.6, 21.2, 23.9, 25.3, 26.0, 26.4, 25.1,
        24.7, 22.9, 25.8, 26.6, 27.2, 23.1, 22.4, 24.0, 21.6, 23.3,
        26.1, 22.0, 21.8, 25.4, 24.2, 26.8, 22.6, 23.0, 27.5, 27.9]
week = ["토", "일", "월", "화", "수", "목", "금"]          # 2024년 6월 1일은 토요일

df = pd.DataFrame({
    "날짜": list(range(1, 31)),
    "요일": [week[i % 7] for i in range(30)],
    "대여건수": rentals,
    "평균기온": temp,
})

front = df.head(15)          # 6/1 ~ 6/15
back = df.tail(15)           # 6/16 ~ 6/30

print("앞 보름:", front.shape, "/ 뒤 보름:", back.shape)
print("앞 보름 평균:", front["대여건수"].mean())
print("뒤 보름 평균:", back["대여건수"].mean())

## Part B. 그림을 둘 — `plt.subplots(1, 2)`

`plt.subplots`의 괄호에 숫자 둘을 주면 **몇 행 몇 열**로 만들지 정합니다.
`(1, 2)`는 1행 2열 — 좌우로 둘입니다. 그리는 문법은 V1·V2에서 배운 `ax.` 그대로입니다. 빈칸 없음.

In [ ]:
fig1, (a, b) = plt.subplots(1, 2, figsize=(11, 4))

a.bar(front["날짜"], front["대여건수"])
b.bar(back["날짜"], back["대여건수"])

a.set_title("June 1-15")
b.set_title("June 16-30")
a.set_xlabel("Day")
b.set_xlabel("Day")
a.set_ylabel("Rentals")

fig1.tight_layout()
print("만들어진 Axes:", type(a).__name__, type(b).__name__, "/ 왼쪽 막대:", len(a.patches), "개")
plt.show()

두 그림의 막대가 **비슷한 높이**로 보입니다.
그런데 Part A에서 본 두 평균은 꽤 차이가 났습니다. 그림이 그 차이를 보여 주고 있나요?

## Part C. 축의 천장을 둘 다 찍어 보기

V1에서 배운 습관대로 축 범위를 찍습니다. **그림이 둘이니 두 번** 찍습니다. 빈칸 없음.

In [ ]:
lo_a, hi_a = a.get_ylim()
lo_b, hi_b = b.get_ylim()

print("왼쪽 y축:", lo_a, hi_a)
print("오른쪽 y축:", lo_b, hi_b)
print("바닥이 둘 다 0 인가:", lo_a == 0.0 and lo_b == 0.0)

바닥은 둘 다 0 입니다 — V1이 가르친 점검을 **통과합니다.**
그런데 **천장이 다릅니다.**

## Part D. 손으로 먼저 — 천장은 어디서 왔을까

두 천장을 각 반달의 값에서 찾아보세요. matplotlib은 막대가 화면 꼭대기에 닿지 않도록
**위에 여백을 조금** 둡니다. 그 여백이 몇 %인지는 Part C의 두 숫자를 각 반달의 **가장 큰 값**과
견주면 나옵니다(둘 다 같은 배수입니다).

- **빈칸 1:** 그 반달에서 **가장 큰 값**을 돌려주는 메서드
- **빈칸 2:** 그 값에 곱해야 천장이 되는 수

In [ ]:
def auto_top(values):
    return values.___() * ___          # ✍️ 빈칸 1·2

print("앞 보름 천장(손계산):", auto_top(front["대여건수"]), "/ 실제:", hi_a)
print("뒤 보름 천장(손계산):", auto_top(back["대여건수"]), "/ 실제:", hi_b)

## Part E. 세트피스 ⭐ — 가장 높은 막대는 화면의 몇 %인가

막대가 화면에서 차지하는 높이는 **값 ÷ 그 Axes의 천장**입니다. 백분율로 보려면 100을 곱합니다.

왼쪽 그림에서 가장 높은 막대는 **6/8의 16420** 입니다.
손으로 계산해 아래 빈칸에 적으세요(**소수점 아래 한 자리**). 그다음 셀을 실행합니다.

- **빈칸 3:** 왼쪽 6/8 막대의 화면 높이(%)

In [ ]:
hand_left = ___                        # ✍️ 빈칸 3

left_top = round(16420 / hi_a * 100, 1)
right_top = round(13880 / hi_b * 100, 1)
print("왼쪽 가장 높은 막대(16420):", left_top, "%")
print("오른쪽 가장 높은 막대(13880):", right_top, "%")
print("손계산과 맞나:", hand_left == left_top)

🔴 **두 수가 같습니다.** 우연이 아닙니다 — 천장이 "최대값 × 그 배수"이니
**가장 높은 막대는 어느 Axes에서든 화면의 같은 높이**에 섭니다. 값이 16420이든 13880이든요.

## Part F. 값과 화면이 거꾸로인 자리 세기

왼쪽 6/1의 값은 **14230** 입니다. 오른쪽 그림에서 **그것과 같은 높이**로 보이는 값은
`오른쪽 천장 × (14230 ÷ 왼쪽 천장)` 입니다. 그보다 크면서 14230보다는 작은 값이 있다면,
그 날은 **값이 더 작은데 화면에서는 더 높아** 보입니다.

뒤 보름 값은 이렇습니다 — 손으로 세어 보세요.

```
5120 7340 10210 4050 8760 12480 5430 5970 11390 9880 11390 4610 7890 13240 13880
```

- **빈칸 4:** 그런 날이 몇 개인가

In [ ]:
guess_taller = ___                     # ✍️ 빈칸 4

line = hi_b * (14230 / hi_a)
taller = [v for v in back["대여건수"] if v > line and v < 14230]
print("6/1(14230)과 같은 높이로 보이는 오른쪽 값:", round(line, 1))
print("값은 작은데 더 높아 보이는 날:", taller)
print("손으로 센 개수와 맞나:", guess_taller == len(taller))

In [ ]:
front_mean = front["대여건수"].mean()
back_mean = back["대여건수"].mean()
print("평균 —— 앞 보름:", front_mean, "/ 뒤 보름:", back_mean)
print("실제 비:", round(back_mean / front_mean * 100, 1), "%")
print("화면에서의 비:", round((back_mean / hi_b) / (front_mean / hi_a) * 100, 1), "%")

실제 차이보다 **화면의 차이가 작습니다.** 장마의 일부가 그림에서 조용히 지워졌습니다.

**왜 그런가:** matplotlib이 축 범위를 정할 때 보는 것은 **그 Axes에 그려진 데이터뿐**입니다.
옆 칸에 무엇이 있는지는 보지 않습니다. 나란히 놓은 것은 **나**이지 matplotlib이 아닙니다.

## Part G. 고치기 — 같은 자로 재기

`subplots`에 인자 **하나**만 더 주면 두 Axes가 y축을 나눠 씁니다. 빈칸 없음.

In [ ]:
fig2, (c, d) = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

c.bar(front["날짜"], front["대여건수"])
d.bar(back["날짜"], back["대여건수"])
c.set_title("June 1-15")
d.set_title("June 16-30")
c.set_ylabel("Rentals")

fig2.tight_layout()

lo_c, hi_c = c.get_ylim()
lo_d, hi_d = d.get_ylim()
print("왼쪽 y축:", lo_c, hi_c, "/ 오른쪽 y축:", lo_d, hi_d)
print("두 천장이 같은가:", hi_c == hi_d)
print("가장 높은 막대 —— 왼쪽:", round(16420 / hi_c * 100, 1), "% / 오른쪽:", round(13880 / hi_d * 100, 1), "%")
print("오른쪽 y축 눈금 글자 수:", len(d.get_yticklabels()))
plt.show()

- 두 천장이 같아졌고, 오른쪽 최고 막대가 눈에 띄게 낮아졌습니다.
- 🔴 **오른쪽 y축 눈금 글자가 사라집니다.** 같은 자를 쓰니 두 번 적을 필요가 없어서입니다. 오류가 아닙니다.
  다만 **오른쪽 칸만 잘라 발표 자료에 붙이면 눈금 없는 막대 그림**이 되니 조심하세요.

## Part H. 다른 길로 고쳐 보기 — "축을 0부터"

V1은 "y축이 0부터인지 보라"고 가르쳤습니다. 그러니 이렇게 해 보고 싶어집니다.
Part C에서 각 그림의 눈금에서 읽은 천장을 그대로 적어 넣고, 바닥을 0으로 못 박습니다. 빈칸 없음.

In [ ]:
fig3, (e, f) = plt.subplots(1, 2, figsize=(11, 4))     # Part B 와 똑같이 그린 뒤
e.bar(front["날짜"], front["대여건수"])
f.bar(back["날짜"], back["대여건수"])
e.set_title("June 1-15")
f.set_title("June 16-30")
e.set_xlabel("Day")
f.set_xlabel("Day")
e.set_ylabel("Rentals")
fig3.tight_layout()

e.set_ylim(0, hi_a)            # 왼쪽 축을 0부터
f.set_ylim(0, hi_b)            # 오른쪽 축도 0부터

lo_e, hi_e = e.get_ylim()
lo_f, hi_f = f.get_ylim()
print("왼쪽 y축:", lo_e, hi_e, "/ 오른쪽 y축:", lo_f, hi_f)
print("Part C 와 똑같은가:", (lo_e, hi_e) == (lo_a, hi_a) and (lo_f, hi_f) == (lo_b, hi_b))
print("두 천장이 같은가:", hi_e == hi_f)
plt.show()

🔴 **두 줄을 더 쳤는데 아무것도 안 바뀌었습니다.** 오류도 경고도 없습니다.

- **이미 0부터였기 때문**입니다. Part C에서 바닥이 둘 다 0인 것을 이미 봤습니다.
- 천장에 넣은 값은 **각 그림에서 읽어 온 값**이라 원래 값을 되돌려 준 셈입니다.
- 손으로 맞추려면 **두 곳에 같은 숫자**를 줘야 합니다 — 그것이 `sharey=True`가 대신 해 주는 일입니다.

> **V1의 점검은 필요하지만 충분하지 않습니다.**
> 그림 하나를 볼 때는 "바닥이 0인가", 그림 둘을 견줄 때는 **"천장도 같은가"** 가 하나 더 붙습니다.

손으로 맞추는 쪽도 한 번 쳐 봅니다 — **두 곳에 같은 숫자**를 줍니다. 빈칸 없음.

In [ ]:
fig3b, (g, h) = plt.subplots(1, 2, figsize=(11, 4))
g.bar(front["날짜"], front["대여건수"])
h.bar(back["날짜"], back["대여건수"])
g.set_title("June 1-15")
h.set_title("June 16-30")
g.set_ylabel("Rentals")

g.set_ylim(0, hi_a)
h.set_ylim(0, hi_a)            # 오른쪽에도 같은 값 — sharey=True 가 대신 해 주던 일이다
fig3b.tight_layout()

print("이제 두 천장이 같은가:", g.get_ylim()[1] == h.get_ylim()[1])
print("오른쪽 가장 높은 막대:", round(13880 / h.get_ylim()[1] * 100, 1), "%")
print("오른쪽 y축 눈금 글자 수:", len(h.get_yticklabels()))
plt.show()

**자는 Part G와 같아졌습니다** — 오른쪽 최고 막대가 똑같이 80.5%입니다.
🔴 그런데 **눈금 글자는 그대로 남습니다.** `sharey=True`는 "같은 자를 쓰니 한 번만 적자"까지 해 주지만,
`set_ylim`을 두 번 주는 것은 **범위만** 맞추는 일이라서요. 두 그림을 나란히 놓고 견주어 보세요.
그리고 숫자를 두 번 적어야 하니, 데이터가 바뀌면 두 곳을 다 고쳐야 합니다.
🔴 **셀을 나눠 그린 두 그림에는 `sharey`를 줄 수 없으니** 그때는 이 손질밖에 방법이 없습니다.

## Part I. 격자와 꾸미기 — 네 칸에 네 그림

행과 열이 둘 다 2 이상이면 자리 번호를 **둘** 줍니다 — `axes[행][열]`, 번호는 0부터입니다.
설명서 §4에서 본 대로 `axes`는 리스트가 아니라 **numpy 배열**이라 `axes.shape`로 모양을 물어볼 수 있습니다.
그리고 Figure에 **직접** 붙인 글은 `fig.texts`에 담깁니다(칸에 붙인 `set_title`과 따로입니다) — 마지막 줄에서 그 수를 셉니다.

- **빈칸 5:** Figure 전체에 제목을 붙이는 메서드(칸마다 붙는 `set_title`과 다릅니다)

In [ ]:
fig4, axes = plt.subplots(2, 2, figsize=(10, 6))

axes[0][0].bar(front["날짜"], front["대여건수"])
axes[0][0].set_title("Bar")

axes[0][1].plot(front["날짜"], front["대여건수"])
axes[0][1].set_title("Line")

axes[1][0].scatter(df["평균기온"], df["대여건수"])
axes[1][0].set_title("Scatter")

axes[1][1].hist(df["대여건수"], bins=6)
axes[1][1].set_title("Histogram")

fig4.___("Four ways to look at the same June")          # ✍️ 빈칸 5
fig4.tight_layout()

titles = [axes[r][c].get_title() for r in range(2) for c in range(2)]
print("axes 의 모양:", axes.shape, "/ 칸마다 제목:", titles)
print("Figure 에 직접 붙인 글 상자 수:", len(fig4.texts))
plt.show()

## Part J. 🔹심화 — 위아래로 놓고 x축만 나눠 쓰기

대여건수와 기온은 **단위가 달라** 한 축에 못 세웁니다. 위아래로 나눠 놓고 **x축만** 공유합니다.
그러면 **아래 칸에만** x 눈금 글자가 남습니다.

- **빈칸 6:** x축을 나눠 쓰게 하는 인자 이름

In [ ]:
fig5, (top, bottom) = plt.subplots(2, 1, figsize=(9, 5), ___=True)          # ✍️ 빈칸 6

top.bar(df["날짜"], df["대여건수"])
top.set_ylabel("Rentals")

bottom.plot(df["날짜"], df["평균기온"])
bottom.set_ylabel("Temp (C)")
bottom.set_xlabel("Day of June")

fig5.tight_layout()
print("위 그림의 x 눈금 글자 수:", len(top.get_xticklabels()))
print("아래 그림의 x 눈금 글자 수:", len(bottom.get_xticklabels()))
plt.show()

🔴 y축은 **공유하지 않았습니다.** 단위가 다른 값에 `sharey=True`를 주면
기온 막대가 0에 붙어 보이지 않게 됩니다 — 공평해지는 것이 아니라 **하나가 사라집니다.**

## Part K. 오류 읽기 다섯 가지

아래 다섯 줄을 **하나씩 주석을 풀어** 실행하고, 오류 이름과 문구를 읽으세요.
읽었으면 다시 주석으로 막아 두어야 다음 셀이 실행됩니다. 빈칸 없음.

In [ ]:
fig6, axes6 = plt.subplots(2, 2)
fig7, axes7 = plt.subplots(1, 2)
fig8, ax8 = plt.subplots()

# 1) axes6[0].bar([1], [1])          # 2행 2열에서 axes[0] 은 Axes 가 아니라 줄 하나
# 2) ax8[0]                          # 숫자를 안 준 subplots() 는 Axes 가 하나뿐
# 3) axes7[2]                        # subplots(1, 2) 인데 셋째를 찾았다
# 4) fig9, (p, q) = plt.subplots(1, 3)   # 받는 이름의 수가 칸 수와 다르다
# 5) fig8.set_title("hello")         # Figure 의 제목은 set_title 이 아니다

print("다섯 줄을 하나씩 풀어 실행해 보세요. 읽었으면 다시 주석 처리합니다.")
plt.close(fig6); plt.close(fig7); plt.close(fig8)

## Part L. `assert` 자가 채점

전부 맞으면 마지막 문구가 나옵니다. 하나라도 틀리면 그 줄에서 멈춥니다.

In [ ]:
assert auto_top(front["대여건수"]) == hi_a and auto_top(back["대여건수"]) == hi_b
assert hand_left == round(16420 / hi_a * 100, 1) == round(13880 / hi_b * 100, 1)
assert guess_taller == len(taller) and 0 < len(taller) < len(back)
assert hi_c == hi_d and hi_c == (hi_a if hi_a > hi_b else hi_b) and len(d.get_yticklabels()) == 0
assert (lo_e, hi_e) == (lo_a, hi_a) and (lo_f, hi_f) == (lo_b, hi_b) and hi_e != hi_f
assert len(fig4.texts) == 1 and fig4.texts[0].get_position()[1] > 0.9 and axes.shape == (2, 2)
assert len(top.get_xticklabels()) == 0 and len(bottom.get_xticklabels()) > 0

print("전부 맞았습니다. 그림 둘을 견주기 전에는 get_ylim() 을 두 번 찍습니다.")